# Construyendo las Primeras Neuronas desde Cero
**Basado en**: *Neural Networks from Scratch* - Capítulo 2 (Coding Our First Neurons)

**Objetivo**: entender la neurona artificial desde su forma mas elemental.
No hay magia: una neurona suma productos ponderados y añade un sesgo.
Todo lo que harán modelos de millones de parámetros escala desde aquí.

**Estructura del notebook:**
1. Una neurona - Python puro, sin librerias
2. Una capa de neuronas - bucle explícito
3. Tensores, arrays y vectores - vocabulario de trabajo
4. Producto escalar y NumPy - la forma eficiente
5. Batch de datos y producto matricial - como escala a producción
6. Ejercicio de decision

---

**Entorno requerido**: `ai_env` (numpy 1.26, matplotlib 3.8)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('images', exist_ok=True)
%matplotlib inline
print(f'NumPy {np.__version__}')
print('[OK] Entorno listo')

NumPy 1.26.4
[OK] Entorno listo


---
## 1. Una neurona - Python puro

Una neurona artificial recibe entradas, las multiplica por pesos y suma un sesgo.

```
output = x1*w1 + x2*w2 + x3*w3 + bias
```

- **Entradas (inputs)**: los datos que llegan a la neurona.
  En producción, serán features del cliente, valores de sensores, tokens de texto...
- **Pesos (weights)**: cuanto importa cada entrada. Son los parámetros que el entrenamiento ajustará.
- **Sesgo (bias)**: un valor adicional que desplaza la salida.
  Sin sesgo, si todas las entradas son 0, la salida siempre sería 0.

> Los pesos y el sesgo son los únicos valores que cambian durante el entrenamiento.
> El resto (la estructura, el número de capas) lo decide el arquitecto del modelo.

In [2]:
# Neurona con 3 entradas - Python puro, sin NumPy
# Los valores son inventados; en un modelo real los aprende el entrenamiento

inputs  = [1, 2, 3]
weights = [0.2, 0.8, -0.5]
bias    = 2

output = (inputs[0]*weights[0] +
          inputs[1]*weights[1] +
          inputs[2]*weights[2] + bias)

print(f'output: {output}')  # esperado: 2.3

output: 2.3


In [3]:
# Neurona con 4 entradas - añadimos un input y su peso asociado
# Cada entrada necesita exactamente un peso

inputs  = [1.0, 2.0, 3.0, 2.5]
weights = [0.2, 0.8, -0.5, 1.0]
bias    = 2.0

output = (inputs[0]*weights[0] +
          inputs[1]*weights[1] +
          inputs[2]*weights[2] +
          inputs[3]*weights[3] + bias)

print(f'output: {output}')  # esperado: 4.8

output: 4.8


---
## 2. Una capa de neuronas

Una capa (layer) es un grupo de neuronas que reciben **las mismas entradas**
pero cada una tiene sus propios pesos y sesgo - y por tanto produce su propia salida.

```
          entradas
         /   |   \
     N1    N2    N3      <- 3 neuronas en la capa
      |     |     |
    out1  out2  out3     <- la capa produce un vector de salidas
```

Si tenemos 3 neuronas y 4 entradas, necesitamos:
- 3 conjuntos de 4 pesos (uno por neurona)
- 3 sesgos (uno por neurona)

A esta arquitectura donde cada neurona se conecta a todas las entradas
se le llama **capa densa** o *fully connected layer*.

In [4]:
# Capa de 3 neuronas - código explícito, sin bucles
# Muestra la estructura pero no escala bien

inputs   = [1, 2, 3, 2.5]

weights1 = [0.2, 0.8, -0.5, 1]
weights2 = [0.5, -0.91, 0.26, -0.5]
weights3 = [-0.26, -0.27, 0.17, 0.87]

bias1 = 2
bias2 = 3
bias3 = 0.5

outputs = [
    # Neurona 1
    inputs[0]*weights1[0] + inputs[1]*weights1[1] +
    inputs[2]*weights1[2] + inputs[3]*weights1[3] + bias1,
    # Neurona 2
    inputs[0]*weights2[0] + inputs[1]*weights2[1] +
    inputs[2]*weights2[2] + inputs[3]*weights2[3] + bias2,
    # Neurona 3
    inputs[0]*weights3[0] + inputs[1]*weights3[1] +
    inputs[2]*weights3[2] + inputs[3]*weights3[3] + bias3,
]

print(outputs)  # esperado: [4.8, 1.21, 2.385]

[4.8, 1.21, 2.385]


In [5]:
# Capa de 3 neuronas - con bucle, escalable a cualquier tamaño
# Los pesos son ahora una lista de listas (una lista por neurona)

inputs  = [1, 2, 3, 2.5]
weights = [
    [0.2,   0.8,   -0.5,  1],     # pesos neurona 1
    [0.5,  -0.91,   0.26, -0.5],  # pesos neurona 2
    [-0.26, -0.27,  0.17,  0.87], # pesos neurona 3
]
biases = [2, 3, 0.5]

layer_outputs = []
for neuron_weights, neuron_bias in zip(weights, biases):
    neuron_output = 0
    for n_input, weight in zip(inputs, neuron_weights):
        neuron_output += n_input * weight
    neuron_output += neuron_bias
    layer_outputs.append(neuron_output)

print(layer_outputs)  # esperado: [4.8, 1.21, 2.385]

# zip() permite iterar dos listas en paralelo:
# itera sobre (neuron_weights_1, bias_1), (neuron_weights_2, bias_2), ...

[4.8, 1.21, 2.385]


---
## 3. Tensores, Arrays y Vectores

Antes de pasar a NumPy conviene fijar el vocabulario:

| Nombre | Dimensiones | Ejemplo Python |
|--------|-------------|----------------|
| Escalar | 0D | `5` |
| Vector | 1D | `[1, 2, 3]` |
| Matriz | 2D | `[[1,2],[3,4]]` |
| Tensor | N-D | cualquier array N-dimensional |

Una lista de Python puede ser cualquier cosa (mezclada, irregular).
Un **array** requiere que todas sus dimensiones sean homogéneas (mismo tamaño en cada nivel).

**Shape** describe las dimensiones de un array:
- `[1, 2, 3]` → shape `(3,)`
- `[[1,2],[3,4],[5,6]]` → shape `(3, 2)` - 3 filas, 2 columnas
- `[[[1,2],[3,4]],[[5,6],[7,8]]]` → shape `(2, 2, 2)`

En deep learning, los datos siempre son tensores.
El tensor mas comun es la **matriz de entradas**: `(n_muestras, n_features)`.

In [6]:
import numpy as np

# Escalar
escalar = np.array(5)
print(f'Escalar  shape: {escalar.shape}  ndim: {escalar.ndim}')

# Vector (1D)
vector = np.array([1, 2, 3])
print(f'Vector   shape: {vector.shape}  ndim: {vector.ndim}')

# Matriz (2D)
matriz = np.array([[4, 2],
                   [5, 1],
                   [8, 2]])
print(f'Matriz   shape: {matriz.shape}  ndim: {matriz.ndim}  -> 3 filas, 2 columnas')

# Tensor 3D
tensor3d = np.array([[[1,5,6,2], [3,2,1,3]],
                     [[5,2,1,2], [6,4,8,4]],
                     [[2,8,5,3], [1,1,9,4]]])
print(f'Tensor3D shape: {tensor3d.shape}  ndim: {tensor3d.ndim}')

# Una lista irregular NO puede ser array
irregular = [[4,2,3], [5,1]]  # segunda lista tiene 2 elementos, no 3
try:
    arr = np.array(irregular, dtype=float)
    print(f'Irregular shape: {arr.shape}')
except Exception as e:
    # En NumPy moderno se crea un array de objetos (ragged)
    arr = np.array(irregular, dtype=object)
    print(f'Irregular: no es homogenea, shape ambigua -> {arr.shape} (array de objetos)')

Escalar  shape: ()  ndim: 0
Vector   shape: (3,)  ndim: 1
Matriz   shape: (3, 2)  ndim: 2  -> 3 filas, 2 columnas
Tensor3D shape: (3, 2, 4)  ndim: 3
Irregular: no es homogenea, shape ambigua -> (2,) (array de objetos)


---
## 4. Producto escalar y NumPy

El **producto escalar** de dos vectores es la suma de los productos de sus elementos:

```
a = [1, 2, 3]
b = [2, 3, 4]
a · b = 1*2 + 2*3 + 3*4 = 2 + 6 + 12 = 20
```

Cuando renombramos `a = inputs` y `b = weights`, el producto escalar hace exactamente
lo que necesitamos: suma los inputs ponderados por sus pesos.

NumPy implementa esto con `np.dot()`, que delega a librerias BLAS/LAPACK compiladas en C.
La diferencia de velocidad frente a un bucle Python es de 10x-100x para arrays grandes.

**Para desarrolladores .NET:** es la diferencia entre un `foreach` y
`System.Numerics.Vector` o `LINQ.Sum()` sobre una coleccion vectorizada.

In [7]:
# Producto escalar manual
a = [1, 2, 3]
b = [2, 3, 4]

dot_manual = a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
print(f'Manual:  {dot_manual}')  # 20

dot_numpy = np.dot(a, b)
print(f'NumPy:   {dot_numpy}')   # 20

Manual:  20
NumPy:   20


In [8]:
# Neurona simple con NumPy
# np.dot(weights, inputs) reemplaza el bucle entero

inputs  = [1.0, 2.0, 3.0, 2.5]
weights = [0.2, 0.8, -0.5, 1.0]
bias    = 2.0

output = np.dot(weights, inputs) + bias
print(f'output: {output}')  # 4.8

output: 4.8


In [9]:
# Capa de 3 neuronas con NumPy
# weights es ahora una matriz (3, 4) - 3 neuronas x 4 entradas
# np.dot(weights_matrix, inputs_vector) devuelve un vector de 3 outputs

inputs  = [1.0, 2.0, 3.0, 2.5]
weights = [
    [0.2,   0.8,   -0.5,  1.0],
    [0.5,  -0.91,   0.26, -0.5],
    [-0.26, -0.27,  0.17,  0.87],
]
biases = [2.0, 3.0, 0.5]

layer_outputs = np.dot(weights, inputs) + biases
print(f'outputs: {layer_outputs}')  # [4.8   1.21  2.385]

# Nota: np.dot(matrix, vector) trata la matrix como lista de vectores fila
# y calcula el producto escalar de cada fila con el vector de inputs.

outputs: [4.8   1.21  2.385]


---
## 5. Batch de datos y producto matricial

Hasta ahora hemos pasado **una sola muestra** por la red.
En producción, las redes procesan **batches** (lotes de muestras) por dos razones:

1. **Velocidad**: las GPUs son paralelas - procesar 32 muestras a la vez
   es casi tan rápido como procesar 1.
2. **Generalización**: el gradiente calculado sobre un batch es mas estable
   que el de una sola muestra. El modelo ajusta pesos hacia el patrón global,
   no hacia una sola observación.

Cuando tanto `inputs` como `weights` son matrices, necesitamos el **producto matricial**:

```
inputs:  (n_muestras, n_features)   -> (3, 4)
weights: (n_features, n_neuronas)   -> necesitamos transponer: (4, 3)
output:  (n_muestras, n_neuronas)   -> (3, 3)
```

La **traspuesta** convierte filas en columnas y columnas en filas.
En NumPy: `np.array(weights).T` o simplemente `weights.T` si ya es un array.

Por convencion, `np.dot(inputs, weights.T)` es el orden estándar:
el resultado tiene shape `(n_muestras, n_neuronas)` - una fila por muestra.

In [10]:
# Traspuesta: filas se convierten en columnas

a = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f'Original   shape: {a.shape}')
print(a)

print(f'\nTraspuesta shape: {a.T.shape}')
print(a.T)

# Propiedad importante: dot product de dos vectores = producto matricial row x column
a_vec = np.array([[1, 2, 3]])    # row vector:    shape (1, 3)
b_vec = np.array([[2, 3, 4]]).T  # column vector: shape (3, 1)
print(f'\nDot como producto matricial: {np.dot(a_vec, b_vec)}')

Original   shape: (2, 3)
[[1 2 3]
 [4 5 6]]

Traspuesta shape: (3, 2)
[[1 4]
 [2 5]
 [3 6]]

Dot como producto matricial: [[20]]


In [11]:
# Batch de 3 muestras procesadas por una capa de 3 neuronas
# Cada fila de inputs es una muestra con 4 features

inputs = [
    [1.0,  2.0,  3.0,  2.5],
    [2.0,  5.0, -1.0,  2.0],
    [-1.5, 2.7,  3.3, -0.8],
]
weights = [
    [0.2,   0.8,   -0.5,  1.0],
    [0.5,  -0.91,   0.26, -0.5],
    [-0.26, -0.27,  0.17,  0.87],
]
biases = [2.0, 3.0, 0.5]

# np.dot(inputs, weights.T): producto matricial
# inputs (3,4) x weights.T (4,3) = resultado (3,3)
layer_outputs = np.dot(inputs, np.array(weights).T) + biases
print('Shape del output:', layer_outputs.shape)
print('Output (fila = muestra, columna = neurona):')
print(layer_outputs)
# esperado: [[ 4.8    1.21   2.385]
#            [ 8.9   -1.81   0.2  ]
#            [ 1.41   1.051  0.026]]

Shape del output: (3, 3)
Output (fila = muestra, columna = neurona):
[[ 4.8    1.21   2.385]
 [ 8.9   -1.81   0.2  ]
 [ 1.41   1.051  0.026]]


---

## 6. Arquitectura: red neuronal multicapa con función de activación

Hasta ahora hemos visto cómo una neurona individual y una capa calculan su salida.
Pero una red real apila **varias capas** en serie. La figura siguiente muestra una
red con 3 entradas, una capa oculta de 4 neuronas y una capa de salida de 2 neuronas.

El símbolo **σ** (sigma) dentro de cada neurona representa la **función de activación**:
la función que transforma la salida bruta de la neurona (suma ponderada + sesgo) antes
de pasarla a la siguiente capa. Sin esta función, toda la red colapsaría a una sola
operación lineal sin importar cuántas capas se añadan.

> En la figura: las flechas son los pesos, las circunferencias son neuronas,
> y el símbolo σ indica que cada neurona aplica una transformación no lineal.

In [12]:
# Diagrama de red neuronal multicapa con simbolo sigma en cada neurona
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pathlib

IMG = pathlib.Path('images')
IMG.mkdir(exist_ok=True)

# Arquitectura de la red
layer_sizes = [3, 4, 4, 2]
layer_labels = ['Entrada', 'Oculta 1', 'Oculta 2', 'Salida']
NEURON_R = 0.22
H_GAP = 2.8
V_SCALE = 1.4

COLOR_INPUT  = '#CFE2FF'
COLOR_HIDDEN = '#D1ECE8'
COLOR_OUTPUT = '#FFE0B2'
EDGE_COLOR   = '#555555'

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_aspect('equal')
ax.axis('off')

# Calcular posiciones
positions = {}
max_neurons = max(layer_sizes)

for l_idx, n_neurons in enumerate(layer_sizes):
    x = l_idx * H_GAP
    total_height = (n_neurons - 1) * V_SCALE
    y_start = -total_height / 2
    for n_idx in range(n_neurons):
        y = y_start + n_idx * V_SCALE
        positions[(l_idx, n_idx)] = (x, y)

# Dibujar conexiones
for l_idx in range(len(layer_sizes) - 1):
    for n_src in range(layer_sizes[l_idx]):
        for n_dst in range(layer_sizes[l_idx + 1]):
            x0, y0 = positions[(l_idx, n_src)]
            x1, y1 = positions[(l_idx + 1, n_dst)]
            ax.plot([x0, x1], [y0, y1],
                    color='#AAAAAA', linewidth=0.7, zorder=1, alpha=0.7)

# Dibujar neuronas
for (l_idx, n_idx), (x, y) in positions.items():
    is_input  = (l_idx == 0)
    is_output = (l_idx == len(layer_sizes) - 1)
    color = COLOR_INPUT if is_input else (COLOR_OUTPUT if is_output else COLOR_HIDDEN)
    circle = plt.Circle((x, y), NEURON_R,
                         facecolor=color, edgecolor=EDGE_COLOR,
                         linewidth=1.4, zorder=3)
    ax.add_patch(circle)
    if is_input:
        label = '$x_{' + str(n_idx + 1) + '}$'
        fs = 11
        fw = 'normal'
    else:
        label = r'$\sigma$'
        fs = 14
        fw = 'bold'
    ax.text(x, y, label, ha='center', va='center', fontsize=fs, zorder=4, fontweight=fw)

# Etiquetas de capa
for l_idx, layer_label in enumerate(layer_labels):
    x = l_idx * H_GAP
    y_max = max(positions[(l_idx, n)][1] for n in range(layer_sizes[l_idx]))
    ax.text(x, y_max + NEURON_R + 0.22, layer_label,
            ha='center', va='bottom', fontsize=9.5,
            color='#333333', style='italic')

# Leyenda
legend_elements = [
    mpatches.Patch(facecolor=COLOR_INPUT,  edgecolor=EDGE_COLOR, label='Neurona de entrada'),
    mpatches.Patch(facecolor=COLOR_HIDDEN, edgecolor=EDGE_COLOR, label='Neurona oculta (aplica sigma)'),
    mpatches.Patch(facecolor=COLOR_OUTPUT, edgecolor=EDGE_COLOR, label='Neurona de salida (aplica sigma)'),
]
ax.legend(handles=legend_elements, loc='lower right',
          fontsize=9, framealpha=0.85, edgecolor='#CCCCCC')

# Anotacion
annotation_text = 'output = sigma(w·x + b)'
ax.annotate(
    annotation_text,
    xy=(H_GAP, 0), xytext=(H_GAP - 0.5, -max_neurons * V_SCALE / 2 - 0.55),
    fontsize=9, color='#444444',
    arrowprops=dict(arrowstyle='->', color='#888888', lw=1.2),
    ha='center',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFDE7', edgecolor='#CCCCCC')
)

ax.set_xlim(-NEURON_R * 3, (len(layer_sizes) - 1) * H_GAP + NEURON_R * 3)
ax.set_ylim(-max_neurons * V_SCALE / 2 - 1.2, max_neurons * V_SCALE / 2 + 0.9)

plt.title('Red neuronal multicapa - arquitectura 3-4-4-2\n'
          'Cada neurona (sigma) aplica una funcion de activacion no lineal',
          fontsize=11, pad=12)
plt.tight_layout()
plt.savefig(IMG / 'B03A_fig02.png', dpi=130, bbox_inches='tight')
plt.close()
print("Guardado: B03A_fig02.png")


Guardado: B03A_fig02.png


In [13]:
# Visualizacion: operacion de una neurona sobre todas las muestras del batch

np.random.seed(42)
n_samples  = 50
n_features = 3
n_neuronas = 4

X = np.random.randn(n_samples, n_features)    # batch de 50 muestras
W = np.random.randn(n_features, n_neuronas)   # pesos: (features, neuronas)
b = np.zeros(n_neuronas)                       # sesgos a cero por ahora

outputs = X @ W + b  # @ es el operador de producto matricial en Python 3
print(f'Input shape:   {X.shape}')
print(f'Weights shape: {W.shape}')
print(f'Output shape:  {outputs.shape}')

# Distribucion de outputs de la neurona 0 sobre el batch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X[:, 0], bins=15, color='steelblue', alpha=0.7, label='feature 0')
axes[0].hist(X[:, 1], bins=15, color='coral', alpha=0.7, label='feature 1')
axes[0].set_title('Distribucion de entradas (batch)')
axes[0].set_xlabel('Valor')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i in range(n_neuronas):
    axes[1].hist(outputs[:, i], bins=12, alpha=0.5, label=f'neurona {i}')
axes[1].set_title('Distribucion de outputs por neurona (sin activacion)')
axes[1].set_xlabel('Valor')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Una capa lineal redistribuye los datos - sin activacion no lineal', fontsize=10)
plt.tight_layout()
plt.savefig('images/B03A_fig01.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close('all')

Input shape:   (50, 3)
Weights shape: (3, 4)
Output shape:  (50, 4)


C:\Users\amado\AppData\Local\Temp\ipykernel_126440\4006787651.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Ejercicio de decision

### Caso: clasificación de tickets de soporte en la empresa

El equipo de soporte recibe 400 tickets diarios.
Actualmente los clasifica un analista que lee cada ticket y asigna categoria.
Un consultor propone entrenar una neurona (o capa densa) para automatizarlo.

**Features disponibles** (por ticket):
- longitud del texto (número de palabras)
- número de adjuntos
- hora de envio
- número de tickets previos del mismo cliente
- si menciona palabras clave como "urgente" o "critico"

**Clases objetivo**: bug / feature request / pregunta de configuración

---

**Pregunta 1**: ¿cuántas entradas (inputs) necesita la neurona?

**Pregunta 2**: Si usaramos una sola neurona (sin capa de salida multiclase),
¿que limitación tendriamos?

**Pregunta 3**: ¿Que información necesitarias para saber si este problema es
resoluble con este approach? ¿Que podria salir mal?

**Pregunta 4**: ¿Hay criterios en los que NO usarias ML para este problema?

---
*Escribe tus respuestas en la celda siguiente.*

### Mis respuestas

**Pregunta 1 - Número de entradas:**

*(escribe aqui)*

**Pregunta 2 - Limitación de una sola neurona:**

*(escribe aqui)*

**Pregunta 3 - Información necesaria:**

*(escribe aqui)*

**Pregunta 4 - Cuando NO usar ML:**

*(escribe aqui)*

---

<!--
CRITERIOS DE EVALUACION (para el instructor)

Pregunta 1: 5 entradas (una por feature listada). Penalizar respuestas que incluyan
el numero de clases como entradas - esas son las salidas, no las entradas.

Pregunta 2: Una sola neurona produce un escalar. Para 3 clases necesitamos
3 neuronas en la capa de salida (una por clase), luego Softmax para convertir
en probabilidades. La limitación no es el numero de entradas sino la capacidad
de representar multiples clases.

Pregunta 3: Información necesaria:
 - Cuántos ejemplos etiquetados hay disponibles (mínimo hundreds)
 - Calidad de las etiquetas (hay desacuerdo entre analistas?)
 - Si las features son discriminativas (longitud del texto predice la categoria?)
 - Frecuencia de cada clase (desbalanceo)
Que podria salir mal:
 - Etiquetas inconsistentes (analistas diferentes clasifican distinto)
 - Features insuficientes (longitud + hora no discriminan bien)
 - El modelo aprende a predecir siempre la clase mayoritaria

Pregunta 4: Criterios de no-uso de ML:
 - Si las reglas son claras y estables ("tickets con 'PROD-DOWN' = urgente siempre")
 - Si el volumen es bajo y el coste del error es alto (clasificacion manual mas segura)
 - Si no hay datos etiquetados suficientes
 - Si el equipo no puede mantener el modelo (reentrenamiento, drift)
-->

---
## Puntos clave del B03A

1. **Una neurona = suma ponderada + sesgo**: `output = dot(inputs, weights) + bias`.
   Nada mas. La complejidad emerge de apilar muchas.

2. **Una capa densa = grupo de neuronas con las mismas entradas**:
   cada neurona tiene sus propios pesos. La salida es un vector.

3. **NumPy vectoriza las operaciones**: `np.dot(inputs, weights.T) + biases`
   reemplaza bucles explícitos y es 10x-100x mas rápido.

4. **Batches = matrices**: cuando los inputs son una matriz `(n_muestras, n_features)`,
   hay que trasponer los pesos para que las dimensiones encajen.

5. **Los pesos son lo que se aprende**: ahora son aleatorios (o inventados).
   El entrenamiento los ajustará para minimizar el error.

---
**Siguiente notebook** - B03B: Añadir Capas
Construiremos la clase `Layer_Dense`, introduciremos datos no lineales
y veremos por que necesitamos mas de una capa oculta.